# Start-to-PCD Trial Duration — Oncology

Derives median **study start → primary completion date (PCD)** by phase, from the
public ClinicalTrials.gov registry.

**How to run:** click each cell in order and press the play button (or Shift+Enter).
Nothing to install. The download cell takes a few minutes.

Method: industry-sponsored interventional oncology trials, drug or biological
intervention, with **actual** (not estimated) start and primary completion dates.

*Independent analysis. Public registry data only.*


## 1. Settings

Edit anything here, then run the cells below. The defaults are the conservative choices.


In [ ]:
# ---- Inclusion criteria -------------------------------------------------
CONDITION_QUERY  = "cancer OR neoplasm OR tumor OR carcinoma OR leukemia OR lymphoma OR myeloma OR sarcoma OR melanoma"

REQUIRE_ACTUAL_DATES = True   # only trials where BOTH dates are ACTUAL, not estimated
INDUSTRY_ONLY        = True   # exclude academic / investigator-initiated
DRUG_BIOLOGIC_ONLY   = True   # exclude device / behavioural / procedure-only
ALLOWED_PHASES       = ("PHASE1", "PHASE2", "PHASE3")

# Only completed trials have an ACTUAL PCD. Note this creates survivorship bias
# (slow trials still running are excluded) - see the caveats at the end.
OVERALL_STATUS = ["COMPLETED"]

MAX_STUDIES = 20000   # safety cap; raise if you want the full set
PAGE_SIZE   = 1000    # API maximum


## 2. Analysis logic

Parsing, filtering and statistics. Tested against synthetic records before release; you shouldn't need to change anything here.


In [ ]:
"""Core parsing + statistics logic for start-to-PCD derivation.
Kept separate so it can be tested without network access.
"""
from datetime import date, timedelta
import statistics

DAYS_PER_MONTH = 30.4375


def parse_ctgov_date(raw):
    """ClinicalTrials.gov dates are 'YYYY-MM-DD' or 'YYYY-MM'.

    Month-only dates are given day 15 (a neutral midpoint) so that
    mixed-precision pairs are not biased systematically early or late.
    Returns a date, or None if unparseable.
    """
    if not raw or not isinstance(raw, str):
        return None
    parts = raw.strip().split("-")
    try:
        if len(parts) == 3:
            return date(int(parts[0]), int(parts[1]), int(parts[2]))
        if len(parts) == 2:
            return date(int(parts[0]), int(parts[1]), 15)
        if len(parts) == 1 and len(parts[0]) == 4:
            return date(int(parts[0]), 7, 1)  # year-only: mid-year
    except (ValueError, TypeError):
        return None
    return None


def extract_study(study):
    """Flatten one API v2 study record into a plain dict.

    Written defensively: every lookup tolerates a missing branch, so a
    schema change degrades to None rather than raising.
    """
    ps = (study or {}).get("protocolSection", {}) or {}
    ident = ps.get("identificationModule", {}) or {}
    status = ps.get("statusModule", {}) or {}
    design = ps.get("designModule", {}) or {}
    sponsor = ps.get("sponsorCollaboratorsModule", {}) or {}
    conds = ps.get("conditionsModule", {}) or {}
    arms = ps.get("armsInterventionsModule", {}) or {}

    start_struct = status.get("startDateStruct", {}) or {}
    pcd_struct = status.get("primaryCompletionDateStruct", {}) or {}
    lead = sponsor.get("leadSponsor", {}) or {}
    enroll = design.get("enrollmentInfo", {}) or {}

    phases = design.get("phases") or []
    interventions = arms.get("interventions") or []
    itypes = sorted({(i or {}).get("type") for i in interventions if (i or {}).get("type")})

    return {
        "nct_id": ident.get("nctId"),
        "title": ident.get("briefTitle"),
        "overall_status": status.get("overallStatus"),
        "start_date_raw": start_struct.get("date"),
        "start_date_type": start_struct.get("type"),
        "pcd_raw": pcd_struct.get("date"),
        "pcd_type": pcd_struct.get("type"),
        "phases": ",".join(phases),
        "lead_sponsor": lead.get("name"),
        "sponsor_class": lead.get("class"),
        "conditions": "; ".join(conds.get("conditions") or []),
        "intervention_types": ",".join(itypes),
        "enrollment": enroll.get("count"),
    }


def add_duration(rec):
    """Attach parsed dates and start-to-PCD duration in months."""
    s = parse_ctgov_date(rec.get("start_date_raw"))
    p = parse_ctgov_date(rec.get("pcd_raw"))
    rec["start_date"] = s
    rec["pcd"] = p
    if s and p:
        days = (p - s).days
        rec["duration_months"] = round(days / DAYS_PER_MONTH, 2)
    else:
        rec["duration_months"] = None
    return rec


def is_usable(rec, require_actual=True, allowed_phases=("PHASE1", "PHASE2", "PHASE3"),
               industry_only=True, drug_biologic_only=True):
    """Apply the inclusion criteria. Returns (bool, reason_if_excluded)."""
    if rec.get("duration_months") is None:
        return False, "missing_or_unparseable_dates"
    if rec["duration_months"] <= 0:
        return False, "non_positive_duration"
    if rec["duration_months"] > 300:
        return False, "implausible_duration_over_25y"
    if require_actual:
        if rec.get("start_date_type") != "ACTUAL":
            return False, "start_date_not_actual"
        if rec.get("pcd_type") != "ACTUAL":
            return False, "pcd_not_actual"
    phases = [p for p in (rec.get("phases") or "").split(",") if p]
    # A trial may be registered as PHASE1,PHASE2 etc.
    if allowed_phases and not any(p in allowed_phases for p in phases):
        return False, "phase_not_in_scope"
    if industry_only and rec.get("sponsor_class") != "INDUSTRY":
        return False, "not_industry_sponsored"
    if drug_biologic_only:
        itypes = set((rec.get("intervention_types") or "").split(","))
        if not ({"DRUG", "BIOLOGICAL"} & itypes):
            return False, "no_drug_or_biological_intervention"
    return True, None


def phase_label(phases_str):
    """Collapse the registry's phase list into a single reporting bucket."""
    phases = [p for p in (phases_str or "").split(",") if p]
    s = set(phases)
    if s == {"PHASE1"}:
        return "Phase 1"
    if s == {"PHASE1", "PHASE2"}:
        return "Phase 1/2"
    if s == {"PHASE2"}:
        return "Phase 2"
    if s == {"PHASE2", "PHASE3"}:
        return "Phase 2/3"
    if s == {"PHASE3"}:
        return "Phase 3"
    if s == {"PHASE4"}:
        return "Phase 4"
    if s == {"EARLY_PHASE1"}:
        return "Early Phase 1"
    return "Other/Unspecified"


def summarise(durations):
    """n, median, mean, Q1, Q3 for a list of durations."""
    vals = sorted(v for v in durations if v is not None)
    if not vals:
        return None
    n = len(vals)
    out = {
        "n": n,
        "median_months": round(statistics.median(vals), 1),
        "mean_months": round(statistics.fmean(vals), 1),
        "min_months": round(vals[0], 1),
        "max_months": round(vals[-1], 1),
    }
    if n >= 4:
        q = statistics.quantiles(vals, n=4, method="inclusive")
        out["q1_months"] = round(q[0], 1)
        out["q3_months"] = round(q[2], 1)
    else:
        out["q1_months"] = None
        out["q3_months"] = None
    return out


## 3. Download from ClinicalTrials.gov

Pages through the public API. Progress prints as it goes.


In [ ]:
import requests, time, json

API = "https://clinicaltrials.gov/api/v2/studies"

FIELDS = ",".join([
    "protocolSection.identificationModule.nctId",
    "protocolSection.identificationModule.briefTitle",
    "protocolSection.statusModule.overallStatus",
    "protocolSection.statusModule.startDateStruct.date",
    "protocolSection.statusModule.startDateStruct.type",
    "protocolSection.statusModule.primaryCompletionDateStruct.date",
    "protocolSection.statusModule.primaryCompletionDateStruct.type",
    "protocolSection.designModule.phases",
    "protocolSection.designModule.enrollmentInfo.count",
    "protocolSection.sponsorCollaboratorsModule.leadSponsor.name",
    "protocolSection.sponsorCollaboratorsModule.leadSponsor.class",
    "protocolSection.conditionsModule.conditions",
    "protocolSection.armsInterventionsModule.interventions.type",
])

def fetch_all(max_studies=MAX_STUDIES):
    studies, token, page = [], None, 0
    while len(studies) < max_studies:
        params = {
            "query.cond": CONDITION_QUERY,
            "filter.overallStatus": "|".join(OVERALL_STATUS),
            "pageSize": PAGE_SIZE,
            "fields": FIELDS,
            "countTotal": "true",
        }
        if token:
            params["pageToken"] = token
        for attempt in range(4):
            try:
                r = requests.get(API, params=params, timeout=90)
                if r.status_code == 200:
                    break
                print("  HTTP", r.status_code, "- retrying...")
            except Exception as e:
                print("  network error:", e, "- retrying...")
            time.sleep(3 * (attempt + 1))
        else:
            print("Giving up after repeated failures.")
            break

        data = r.json()
        batch = data.get("studies", [])
        if not batch:
            break
        studies.extend(batch)
        page += 1
        if page == 1 and data.get("totalCount"):
            print("Total matching the query:", data["totalCount"])
        print("  page %d - %d studies so far" % (page, len(studies)))
        token = data.get("nextPageToken")
        if not token:
            break
        time.sleep(0.5)
    return studies

raw_studies = fetch_all()
print("\nDownloaded", len(raw_studies), "study records.")


## 4. Parse, filter, and see what was excluded

The exclusion counts matter — if most trials drop out for one reason, that tells you something about the data.


In [ ]:
import pandas as pd
from collections import Counter

records, reasons = [], Counter()
for s in raw_studies:
    rec = add_duration(extract_study(s))
    ok, why = is_usable(
        rec,
        require_actual=REQUIRE_ACTUAL_DATES,
        allowed_phases=ALLOWED_PHASES,
        industry_only=INDUSTRY_ONLY,
        drug_biologic_only=DRUG_BIOLOGIC_ONLY,
    )
    if ok:
        rec["phase_label"] = phase_label(rec["phases"])
        records.append(rec)
    else:
        reasons[why] += 1

df = pd.DataFrame(records)
print("Included:", len(df), "trials\n")
print("Excluded, by reason:")
for reason, n in reasons.most_common():
    print("  %-40s %d" % (reason, n))


## 5. The result — median start-to-PCD by phase

This is the table for the model.


In [ ]:
rows = []
order = ["Phase 1", "Phase 1/2", "Phase 2", "Phase 2/3", "Phase 3"]
for label in order:
    sub = df[df["phase_label"] == label]
    st = summarise(sub["duration_months"].tolist())
    if st:
        rows.append({"Phase": label, **st})

summary = pd.DataFrame(rows)[
    ["Phase", "n", "median_months", "q1_months", "q3_months", "mean_months", "min_months", "max_months"]
]
print("Start-to-PCD duration, oncology, industry-sponsored (months)\n")
display(summary)

overall = summarise(df["duration_months"].tolist())
print("\nAll included phases: n=%d, median=%.1f months" % (overall["n"], overall["median_months"]))


## 6. Distribution plot

Medians hide a lot. The spread is the more honest picture.


In [ ]:
import matplotlib.pyplot as plt

present = [l for l in order if (df["phase_label"] == l).any()]
data = [df[df["phase_label"] == l]["duration_months"].tolist() for l in present]

fig, ax = plt.subplots(figsize=(9, 5))
try:
    bp = ax.boxplot(data, tick_labels=present, patch_artist=True, showfliers=False)
except TypeError:  # matplotlib < 3.9
    bp = ax.boxplot(data, labels=present, patch_artist=True, showfliers=False)
for patch in bp["boxes"]:
    patch.set_facecolor("#0d9488")
    patch.set_alpha(0.65)
for median in bp["medians"]:
    median.set_color("#0c1e2e")
    median.set_linewidth(2)
ax.set_ylabel("Months from study start to primary completion")
ax.set_title("Start-to-PCD duration by phase - oncology, industry-sponsored")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Sponsor split (optional sanity check)

Re-run with `INDUSTRY_ONLY = False` in cell 1 first if you want to compare against academic trials.


In [ ]:
if "sponsor_class" in df.columns and df["sponsor_class"].nunique() > 1:
    for cls, sub in df.groupby("sponsor_class"):
        st = summarise(sub["duration_months"].tolist())
        if st and st["n"] >= 10:
            print("%-12s n=%-6d median=%.1f months" % (cls, st["n"], st["median_months"]))
else:
    print("Only one sponsor class in the current filter (INDUSTRY_ONLY is on).")


## 8. Export

Downloads two CSVs: the summary table, and the full trial-level dataset so anyone can reproduce or audit the result.


In [ ]:
summary.to_csv("start_to_pcd_summary.csv", index=False)
df.to_csv("start_to_pcd_trials.csv", index=False)
print("Wrote start_to_pcd_summary.csv (%d rows)" % len(summary))
print("Wrote start_to_pcd_trials.csv (%d rows)" % len(df))

try:
    from google.colab import files
    files.download("start_to_pcd_summary.csv")
    files.download("start_to_pcd_trials.csv")
except Exception:
    print("(Not running in Colab - files saved alongside the notebook.)")


## Caveats to state when you publish this

1. **Survivorship bias.** Only completed trials have an actual PCD, so trials still
   running — disproportionately the slow ones — are excluded. These medians are
   therefore likely to run short. This is the most important limitation.
2. **Date precision.** Older registry records hold month-and-year only. Month-only
   dates are assigned day 15 here, so individual durations carry up to about two
   weeks of noise either way. It cancels out across a large sample.
3. **Amendment history is invisible.** The API returns current values. Sponsors
   revise PCD over time, and those revisions aren't visible in a single snapshot.
4. **Pivotal trials aren't flagged.** The registry can't tell you which trial supported
   an approval. "Industry-sponsored Phase 3 oncology" is a proxy for the pivotal
   population, not the population itself.
5. **Condition matching is text-based**, not MeSH-tree based, so the oncology set is
   approximate at the edges.
6. **PCD is endpoint-dependent.** Where the primary endpoint is survival, start-to-PCD
   approaches full trial duration; where it's response rate, it's much shorter. A
   single median per phase hides that split.

**Data source:** ClinicalTrials.gov public API v2, accessed on the date you run this.
Query and filters are recorded in cell 1.
